# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected.")

Connected.


In [7]:
TABLES['fact_daily_march'] = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
n = con.sql(f"SELECT COUNT(*) FROM {TABLES['fact_daily_march']}").fetchone()[0]
print(f"March partition row count: {n:,}")

March partition row count: 9,841,378


**Unit of analysis:** One row = one content page (`content_hash_id`),
within one client (`client_hash_id`), on one calendar day
(`report_date`), from `fact_content_daily_performance`.

**Time window:** `month=2026-03` (a mid-panel month) — deliberately
avoiding the final month (2026-06), which is the sealed test month and
the natural outcome window for any past→future label.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (candidates, knowable before the decision point):**
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `gsc_sum_position`
  - search visibility signals, available for 36.7% of March rows
- `ga4_pageviews`, `ga4_sessions`, `ga4_engaged_sessions`,
  `ga4_total_engagement_sec` - engagement signals, but only available
  for 4.2% of rows (see Section 3 finding), so these will need careful
  handling (many missing values) if used
- `sessions_organic`, `sessions_direct`, `sessions_referral`,
  `sessions_social`, `sessions_paid`, `sessions_ai` - traffic-source
  breakdown
- `scroll_events` - on-page engagement signal

**Label/proxy:** A declining-impressions flag, built by comparing a
last-N-day window of `gsc_impressions` to a prior-N-day window within
the slice - not yet built, will construct this the same way as
notebook 03's example (last30 vs prev30).

**Context (join keys and availability flags, not features):**
`report_date`, `client_hash_id`, `content_hash_id`, `client_has_gsc`,
`client_has_ga4`, `gsc_data_available`, `ga4_data_available`, `month`.

**Excluded, and why:**
- Any FlyRank product decision flag (`health_score`, `priority_score`,
  `action_type`) - not present in this table by design, and I won't
  reconstruct them.
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`,
  `ai_meta`, `ai_other` - AI-referral breakdown columns exist, but per
  the lane guide these are sparse and only safe for EDA/ranking, not
  as classifier features.
- Any column that is itself derived from a future outcome relative to
  whatever window I define for my label, since that would leak the
  answer.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily_march']} LIMIT 1").df()
print(cols['column_name'].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries: grain uniqueness, row count/date span, and availability
filtered with IS TRUE.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_daily_march']}
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
""").df()
print("Grain violations (should be 0):", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (should be 0): 0


In [13]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily_march']}
""").df()
print(span)

    n_rows   min_date   max_date
0  9841378 2026-03-01 2026-03-31


In [14]:
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
    FROM {TABLES['fact_daily_march']}
""").df()
print(avail)

   total_rows  ga4_available_rows  gsc_available_rows
0     9841378              413966             3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Unbalanced panel:** History depth differs per client - some have
  12+ months, others much less.
- **GA4/GSC availability gap (observed above):** Only ~4.2% of March
  rows have GA4 data, versus ~36.7% for GSC - any GA4-based feature
  will have far fewer usable rows.
- **Window overlap risk:** If I build a prior→next window label,
  feature and label windows must never overlap, or that's leakage.
- **Anonymized query tail:** rare/anonymized queries in the query-level
  table limit how precise query-based features can get.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
clients = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()
print(clients.head(10))

            client_hash_id gsc_data_start ga4_data_start
0  client_9958f0a7ae1df715     2025-01-27     2025-10-29
1  client_ff644d8251367cbb     2025-01-27     2025-10-29
2  client_73cda7b4e4f265ea     2025-02-11     2026-03-24
3  client_fef1a8f436438636     2025-03-11     2026-03-06
4  client_62f4a7e64f5e0096     2025-06-07            NaT
5  client_b10cb2997d0c7c86     2025-06-18     2025-11-15
6  client_65de48885f4ef01b     2025-06-21     2026-02-19
7  client_c182d11e4862a37d     2025-06-21     2026-02-20
8  client_3197e6291363b4db     2025-06-29     2025-11-09
9  client_625b6439094e23e4     2025-07-01     2026-02-19


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Self-check**

- Every section above is filled - markdown thinking AND the code that
  backs it: Yes
- The notebook runs top to bottom with no errors (Runtime → Run all):
  Yes, confirmed
- No client names, URLs, or private queries anywhere: Confirmed - only
  pseudonymized hash IDs used throughout
- My claims use careful words: observed, measured, directional,
  decision-support: Yes - e.g. "observed above," row counts and
  percentages stated as measured facts, not predictions
- Committed to my repo under `work/notebooks/` - then submitted repo
  URL on the card: Done